In [1]:
import nflreadpy as nfl
import pandas as pd

print("Bajando datos de partidos pasados...")
raw_data = nfl.load_schedules([2023, 2024])

games = raw_data.to_pandas()

cols = games.columns.tolist()
print("Las columnas disponibles son:", cols[:15]) 

score_cols = [c for c in cols if 'score' in c.lower()]
print(f"Columnas de puntos detectadas: {score_cols}")

games_clean = games.dropna(subset=score_cols)

print(f"¡Listo! Tenemos {len(games_clean)} partidos para estudiar.")
games_clean[['season', 'week', 'home_team', 'away_team'] + score_cols].head()

Bajando datos de partidos pasados...
Las columnas disponibles son: ['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime']
Columnas de puntos detectadas: ['away_score', 'home_score']
¡Listo! Tenemos 570 partidos para estudiar.


,season,week,home_team,away_team,away_score,home_score
0,2023,1,KC,DET,21,20
1,2023,1,ATL,CAR,10,24
2,2023,1,BAL,HOU,9,25
3,2023,1,CLE,CIN,3,24
4,2023,1,IND,JAX,31,21


In [2]:
import pandas as pd

df_analytics = pd.read_csv('NFL Power Rankings 2025 + Analytics.csv')

map_teams = {
    'ARI': 'Arizona Cardinals', 'ATL': 'Atlanta Falcons', 'BAL': 'Baltimore Ravens', 'BUF': 'Buffalo Bills',
    'CAR': 'Carolina Panthers', 'CHI': 'Chicago Bears', 'CIN': 'Cincinnati Bengals', 'CLE': 'Cleveland Browns',
    'DAL': 'Dallas Cowboys', 'DEN': 'Denver Broncos', 'DET': 'Detroit Lions', 'GB': 'Green Bay Packers',
    'HOU': 'Houston Texans', 'IND': 'Indianapolis Colts', 'JAX': 'Jacksonville Jaguars', 'KC': 'Kansas City Chiefs',
    'LV': 'Las Vegas Raiders', 'LAC': 'Los Angeles Chargers', 'LAR': 'Los Angeles Rams', 'MIA': 'Miami Dolphins',
    'MIN': 'Minnesota Vikings', 'NE': 'New England Patriots', 'NO': 'New Orleans Saints', 'NYG': 'New York Giants',
    'NYJ': 'New York Jets', 'PHI': 'Philadelphia Eagles', 'PIT': 'Pittsburgh Steelers', 'SEA': 'Seattle Seahawks',
    'SF': 'San Francisco 49ers', 'TB': 'Tampa Bay Buccaneers', 'TEN': 'Tennessee Titans', 'WAS': 'Washington Commanders'
}

def get_team_stats(team_abbr, analytics_df):
    full_name = map_teams.get(team_abbr)
    stats = analytics_df[analytics_df['Team (2025) (e21)'] == full_name]
    return stats

print(f"Probando mapeo para KC: {map_teams['KC']}")
print(get_team_stats('KC', df_analytics)[['Team (2025) (e21)', 'EPA/Play (Custom) (8a0)']])

Probando mapeo para KC: Kansas City Chiefs
     Team (2025) (e21)  EPA/Play (Custom) (8a0)
17  Kansas City Chiefs                    0.115


In [ ]:
import pandas as pd

stats_to_use = df_analytics[[
    'Team (2025) (e21)', 
    'EPA/Play (Custom) (8a0)', 
    'Success Rate (Custom) (8a0)', 
    'Yards / Play  (Calculated)'
]].copy()

stats_to_use.columns = ['team_full', 'epa', 'success_rate', 'yards_play']

if stats_to_use['success_rate'].dtype == 'O': 
    stats_to_use['success_rate'] = stats_to_use['success_rate'].str.replace('%', '').astype(float)

inv_map = {v: k for k, v in map_teams.items()}
stats_to_use['abbr'] = stats_to_use['team_full'].map(inv_map)

df_master = pd.merge(games_clean, stats_to_use, left_on='home_team', right_on='abbr', how='left')
df_master = df_master.rename(columns={
    'epa': 'home_epa', 
    'success_rate': 'home_success', 
    'yards_play': 'home_yards'
})

df_master = pd.merge(df_master, stats_to_use, left_on='away_team', right_on='abbr', how='left')
df_master = df_master.rename(columns={
    'epa': 'away_epa', 
    'success_rate': 'away_success', 
    'yards_play': 'away_yards'
})

df_master['result'] = (df_master['home_score'] > df_master['away_score']).astype(int)

df_master = df_master.drop(columns=['abbr_x', 'abbr_y', 'team_full_x', 'team_full_y'])

print("¡Tabla Maestra generada y datos LIMPIOS!")
df_master[['home_team', 'away_team', 'home_epa', 'home_success', 'result']].head()

¡Tabla Maestra generada y datos LIMPIOS!


,home_team,away_team,home_epa,home_success,result
0,KC,DET,0.115,47.2,0
1,ATL,CAR,-0.024,45.2,1
2,BAL,HOU,-0.011,43.8,1
3,CLE,CIN,-0.163,35.1,1
4,IND,JAX,0.128,48.3,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df_ml = df_master.dropna(subset=['home_epa', 'away_epa', 'home_success', 'away_success'])

features = ['home_epa', 'home_success', 'home_yards', 'away_epa', 'away_success', 'away_yards']
X = df_ml[features]
y = df_ml['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

_pred = modelo.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"--- MODELO ENTRENADO ---")
print(f"Precisión del modelo: {accuracy:.2%}")
print("\nReporte detallado:")
print(classification_report(y_test, y_pred))

import pandas as pd
import matplotlib.pyplot as plt

importancias = pd.Series(modelo.feature_importances_, index=features).sort_values(ascending=False)
print("\nImportancia de las variables (¿Qué ayudó más a predecir?):")
print(importancias)

--- MODELO ENTRENADO ---
Precisión del modelo: 64.49%

Reporte detallado:
              precision    recall  f1-score   support

           0       0.59      0.59      0.59        46
           1       0.69      0.69      0.69        61

    accuracy                           0.64       107
   macro avg       0.64      0.64      0.64       107
weighted avg       0.64      0.64      0.64       107



Matplotlib is building the font cache; this may take a moment.



Importancia de las variables (¿Qué ayudó más a predecir?):
home_yards      0.181783
home_epa        0.169404
away_yards      0.168397
away_epa        0.162825
away_success    0.158880
home_success    0.158711
dtype: float64


In [ ]:
print("--------------------------------------------------")
print("🏟️  BIENVENIDO AL SIMULADOR DE LA NFL 🏟️")
print("Instrucciones: Cambia las siglas de abajo por los equipos que quieras.")
print("Ejemplos: KC, SF, DAL, PHI, BUF, DET, etc.")
print("--------------------------------------------------")

equipo_local = 'ARI'     
equipo_visitante = 'SF' 

def simular_partido(home, away):
    
    stats_h = stats_to_use[stats_to_use['abbr'] == home]
    stats_a = stats_to_use[stats_to_use['abbr'] == away]
    
    if stats_h.empty or stats_a.empty:
        return "⚠️ Error: Uno de los equipos no existe o está mal escrito. Revisa las siglas."

    
    partido_input = [[
        stats_h['epa'].values[0], 
        stats_h['success_rate'].values[0], 
        stats_h['yards_play'].values[0],
        stats_a['epa'].values[0], 
        stats_a['success_rate'].values[0], 
        stats_a['yards_play'].values[0]
    ]]

    
    probabilidades = modelo.predict_proba(partido_input)[0] 
    prob_victoria_local = probabilidades[1] * 100
    
    
    print(f"\nSaliendo al campo: {home} vs {away}")
    print(f"Lugar: Estadio de los {map_teams[home]}")
    print("-" * 30)
    
    if prob_victoria_local > 50:
        print(f"🔮 PREDICCIÓN: Ganan los {map_teams[home]} (LOCAL)")
    else:
        print(f"🔮 PREDICCIÓN: Ganan los {map_teams[away]} (VISITANTE)")
        
    print(f"📈 Probabilidad de victoria local: {prob_victoria_local:.1f}%")
    print("-" * 30)


simular_partido(equipo_local, equipo_visitante)

--------------------------------------------------
🏟️  BIENVENIDO AL SIMULADOR DE LA NFL 🏟️
Instrucciones: Cambia las siglas de abajo por los equipos que quieras.
Ejemplos: KC, SF, DAL, PHI, BUF, DET, etc.
--------------------------------------------------

Saliendo al campo: ARI vs SF
Lugar: Estadio de los Arizona Cardinals
------------------------------
🔮 PREDICCIÓN: Ganan los San Francisco 49ers (VISITANTE)
📈 Probabilidad de victoria local: 49.7%
------------------------------


/opt/homebrew/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
